In [ ]:
import yaml
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from landscaper import LossLandscape, PyHessian
from tgn.model.tgn import TGN
from tgn.utils.data import Data, compute_time_statistics
from tgn.utils.utils import get_neighbor_finder

input_dir = Path("outputs/2026-09-30/tgn__MovieLens-1M__01-33-38")
data_dir = Path("data/MovieLens-1M")
device = torch.device("cpu")
model_config = yaml.safe_load((input_dir / ".hydra/config.yaml").read_text())["model"]
batch_size = int(model_config["batch_size"])
memory_batch_size = int(model_config["memory_batch_size"])
assert batch_size % memory_batch_size == 0
n_neighbors = int(model_config["n_degree"])

train_df = pd.read_csv(data_dir / "train.csv").sort_values("timestamp", kind="mergesort").reset_index(drop=True)
source_nodes = train_df["user_id"].to_numpy(np.int64) + 1
destination_nodes = train_df["item_id"].to_numpy(np.int64) + 6040 + 1
timestamps = train_df["timestamp"].to_numpy(np.float64)
edge_ids = np.arange(1, len(train_df) + 1, dtype=np.int64) if model_config.get("edge") else np.zeros(len(train_df), dtype=np.int64)
num_nodes = 6040 + 3662 + 1
node_features = np.zeros((num_nodes, 64), dtype=np.float32)
edge_features = np.zeros((len(train_df) + 1, 64), dtype=np.float32)
edge_features[1:, 0] = 1.0
node_features[0] = 0
edge_features[0] = 0
train_data = Data(
    sources=source_nodes,
    destinations=destination_nodes,
    timestamps=timestamps,
    edge_idxs=edge_ids,
    labels=np.ones(len(train_df), dtype=np.float32),
)
train_neighbor_finder = get_neighbor_finder(train_data, uniform=False)
mean_src, std_src, mean_dst, std_dst = compute_time_statistics(
    source_nodes, destination_nodes, timestamps
)
model = TGN(
    neighbor_finder=train_neighbor_finder,
    node_features=node_features,
    edge_features=edge_features,
    device=device,
    n_layers=1,
    n_heads=2,
    dropout=0.1,
    use_memory=True,
    memory_update_at_start=True,
    message_dimension=100,
    memory_dimension=64,
    embedding_module_type="graph_attention",
    message_function="identity",
    mean_time_shift_src=mean_src,
    std_time_shift_src=max(std_src, 1e-10),
    mean_time_shift_dst=mean_dst,
    std_time_shift_dst=max(std_dst, 1e-10),
    n_neighbors=n_neighbors,
    aggregator_type="mean",
    memory_updater_type="gru",
    use_edge_features=False,
    trainable_node_features=bool(model_config["trainable_node_features"]),
).to(device)

checkpoint = torch.load(input_dir / "tgn_weights.pt", map_location=device)
model.load_state_dict(checkpoint)
model.eval()

split_idx = len(train_data.sources) - 4096

model.memory.__init_memory__()
with torch.no_grad():
    for start in range(0, split_idx, memory_batch_size):
        end = min(start + memory_batch_size, split_idx)
        sources = train_data.sources[start:end]
        destinations = train_data.destinations[start:end]
        timestamps = train_data.timestamps[start:end]
        edge_idxs = train_data.edge_idxs[start:end]
        model.process_real_interaction(source_nodes=sources, destination_nodes=destinations, edge_times=timestamps, edge_idxs=edge_idxs, n_neighbors=n_neighbors)

model.memory.detach_memory()
eval_sources = train_data.sources[split_idx:]
eval_destinations = train_data.destinations[split_idx:]
eval_timestamps = train_data.timestamps[split_idx:]
eval_edge_idxs = train_data.edge_idxs[split_idx:]

rng = np.random.RandomState(42)

positive_item_ids = eval_destinations - 6040 - 1
negative_item_ids = rng.randint(0, 3662, size=len(eval_sources))
collisions = negative_item_ids == positive_item_ids
while collisions.any():
    negative_item_ids[collisions] = rng.randint(0, 3662, size=collisions.sum())
    collisions = negative_item_ids == positive_item_ids

eval_negatives = negative_item_ids + 6040 + 1

model.memory.detach_memory()
memory_backup = model.memory.backup_memory()


def loss_landscaper(model):
    model.memory.restore_memory(memory_backup)
    epoch_losses = []
        
    for start in range(0, len(eval_sources), memory_batch_size):
        end = min(start + memory_batch_size, len(eval_sources))
        sources = eval_sources[start:end]
        positives = eval_destinations[start:end]
        negatives = eval_negatives[start:end]
        times = eval_timestamps[start:end]
        edge_idxs = eval_edge_idxs[start:end]

        source_emb, positive_emb, negative_emb = model.compute_temporal_embeddings(source_nodes=sources, destination_nodes=positives, candidate_nodes=negatives, edge_times=times, edge_idxs=edge_idxs, n_neighbors=n_neighbors)

        pos_scores = torch.sum(source_emb * positive_emb, dim=1)
        neg_scores = torch.sum(source_emb * negative_emb, dim=1)

        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8)
        epoch_losses.append(loss)
        model.memory.detach_memory()
    return torch.cat(epoch_losses).mean()

loss = loss_landscaper(model)
print(loss.item())


In [ ]:
checkpoint_state = {
    k: v.clone()
    for k, v in model.state_dict().items()
}

In [ ]:
values = []

for i in range(20):
    values.append(loss_landscaper(model).item())

print(values)
print("std:", np.std(values))
print("range:", max(values) - min(values))

In [ ]:
def tgn_hessian_generator(model, criterion, data, device):
    params = [p for p in model.parameters() if p.requires_grad]

    model.zero_grad(set_to_none=True)
    current_loss = loss_landscaper(model)
    current_loss = current_loss + sum(
        0.0 * (p ** 2).sum() for p in params
    )

    grads = torch.autograd.grad(
        current_loss,
        params,
        create_graph=True,
    )

    yield len(eval_sources), grads


from torch.nn.attention import sdpa_kernel, SDPBackend

torch.backends.mha.set_fastpath_enabled(False)

with sdpa_kernel(SDPBackend.MATH):

    hessian = PyHessian(
        model=model,
        criterion=None,
        data=None,
        device="cpu",
        hessian_generator=tgn_hessian_generator,
        try_cache=True,
    )

    eigenvalues, trainable_eigenvectors = hessian.eigenvalues(
        top_n=2,
        maxIter=50,
        tol=1e-3,
    )

print("Eigenvalues:", eigenvalues)

params = list(model.parameters())
directions = []

for eigenvector in trainable_eigenvectors:
    eigenvector_iter = iter(eigenvector)
    directions.append([
        next(eigenvector_iter)
        if parameter.requires_grad
        else torch.zeros_like(parameter)
        for parameter in params
    ])

trainable_norm = torch.sqrt(
    sum((p ** 2).sum() for p in params if p.requires_grad)
)
full_norm = torch.sqrt(
    sum((p ** 2).sum() for p in params)
)
distance = 0.01 * (trainable_norm / full_norm).item()

del hessian

def scalar_function(model, data):
    return loss_landscaper(model).item()

landscape = LossLandscape.compute(
    model=model,
    data=None,
    dirs=directions,
    scalar_fn=scalar_function,
    steps=41,
    distance=distance,
    dim=2,
    device="cpu",
)

relative_ranges = [
    r * (full_norm / trainable_norm).item() * 100
    for r in landscape.ranges
]

relative_landscape = LossLandscape(
    landscape.loss,ы
    relative_ranges,
)

import matplotlib.pyplot as plt

fig = relative_landscape.show(show=False)
fig.savefig("tgn_loss_landscape.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
relative_ranges = [
    r * (full_norm / trainable_norm).item() * 100
    for r in landscape.ranges
]

relative_landscape = LossLandscape(
    landscape.loss,
    relative_ranges,
)

fig = relative_landscape.show_contour(show=False)

fig.savefig(
    "tgn_loss_landscape_contour.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
profile_landscape = LossLandscape(
    landscape.loss,
    [r / np.max(np.abs(r)) for r in landscape.ranges],
)

profile_landscape.show_profile()

In [ ]:
model.load_state_dict(checkpoint_state)
model.memory.restore_memory(memory_backup)

In [ ]:
model.memory.restore_memory(memory_backup)
model.memory.detach_memory()
memory_backup = model.memory.backup_memory()

model.zero_grad(set_to_none=True)
params = [p for p in model.parameters() if p.requires_grad]
loss = loss_landscaper(model)

grads = torch.autograd.grad(
    loss,
    params,
    allow_unused=True,
)

grad_norm = torch.sqrt(
    sum((g ** 2).sum() for g in grads if g is not None)
)

param_norm = torch.sqrt(
    sum((p ** 2).sum() for p in params)
)

relative_grad_norm = grad_norm / param_norm

print(f"Градиент: {grad_norm.item()}")
print(f"Относительный градиент: {relative_grad_norm.item()}")